In [ ]:
from zipfile import ZipFile

In [ ]:
with ZipFile("/content/archive2.zip","r") as zip:
  zip.extractall()

In [ ]:
from zipfile import ZipFile, BadZipFile
import os

zip_file_path = "/content/archive2.zip"

if not os.path.exists(zip_file_path):
    print(f"Error: File not found at {zip_file_path}")
elif not os.path.isfile(zip_file_path):
    print(f"Error: Path {zip_file_path} is not a file.")
else:
    try:
        with ZipFile(zip_file_path, "r") as zip_ref:
            print(f"{zip_file_path} is a valid zip file. It contains the following files and directories:")
            for name in zip_ref.namelist():
                print(f"  - {name}")
    except BadZipFile:
        print(f"{zip_file_path} is NOT a valid zip file. It might be corrupted or in an incorrect format.")
    except Exception as e:
        print(f"An unexpected error occurred while checking {zip_file_path}: {e}")

/content/archive2.zip is NOT a valid zip file. It might be corrupted or in an incorrect format.


In [ ]:
with ZipFile("/content/archive2.zip","r") as zip:
  zip.extractall()

In [ ]:
!pip install --upgrade torch torchvision

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 530.7/530.7 MB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.1/366.1 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.9/169.9 MB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.5/196.5 MB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 70.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 52.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.1/214.1 MB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.5/5

In [ ]:
!pip install -U sympy
!pip install -U torch torchvision


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from PIL import Image
import os

# ========================
# 1. Device Setup
# ========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ========================
# 2. Dataset Paths
# ========================
train_dir = "/content/dataset/train"   # adjust path if needed
val_dir   = "/content/dataset/validation"

# ========================
# 3. Data Transforms
# ========================
IMG_SIZE = 224
BATCH_SIZE = 32

train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# ========================
# 4. Dataset & Dataloader
# ========================
train_dataset = datasets.ImageFolder(train_dir, transform=train_transforms)
val_dataset = datasets.ImageFolder(val_dir, transform=val_transforms)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

classes = train_dataset.classes
print("Classes:", classes)  # ['not_out', 'run_out']

# ========================
# 5. Load MobileNetV2
# ========================
model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1)

# Freeze base layers
for param in model.features.parameters():
    param.requires_grad = False

# Replace classifier head
model.classifier[1] = nn.Sequential(
    nn.Linear(model.last_channel, 256),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(256,128),
    nn.ReLU(),
    nn.Dropout(0,3),
    nn.Linear(128, len(classes))  # 2 classes
)

model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

EPOCHS = 25

for epoch in range(EPOCHS):
    model.train()
    running_loss, correct, total = 0, 0, 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    train_acc = 100 * correct / total
    train_loss = running_loss / len(train_loader)

    # Validation
    model.eval()
    val_correct, val_total, val_loss = 0, 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)

    val_acc = 100 * val_correct / val_total
    val_loss /= len(val_loader)

    print(f"Epoch [{epoch+1}/{EPOCHS}] "
          f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}% | "
          f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")


torch.save(model.state_dict(), "mobilenet_runout.pth")
print("Model saved as mobilenet_runout.pth")

def predict_image(img_path):
    model.eval()
    transform = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    image = Image.open(img_path).convert("RGB")
    img_tensor = transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(img_tensor)
        _, predicted = torch.max(outputs, 1)

    return classes[predicted.item()]

test_img = "/content/dataset/validation/out/out (12).jpg"
if os.path.exists(test_img):
    result = predict_image(test_img)
    print(f"Prediction for {test_img}: {result}")
else:
    print("No test image found. Place a test.jpg in working directory.")


Using device: cpu
Classes: ['not_out', 'out']
Epoch [1/25] Train Loss: 0.5863, Train Acc: 72.86% | Val Loss: 0.5850, Val Acc: 70.37%
Epoch [2/25] Train Loss: 0.5484, Train Acc: 73.44% | Val Loss: 0.5510, Val Acc: 70.37%
Epoch [3/25] Train Loss: 0.5015, Train Acc: 75.59% | Val Loss: 0.5055, Val Acc: 76.22%
Epoch [4/25] Train Loss: 0.4802, Train Acc: 76.96% | Val Loss: 0.4863, Val Acc: 76.02%
Epoch [5/25] Train Loss: 0.4505, Train Acc: 78.63% | Val Loss: 0.4646, Val Acc: 75.63%
Epoch [6/25] Train Loss: 0.4521, Train Acc: 78.85% | Val Loss: 0.4575, Val Acc: 76.41%
Epoch [7/25] Train Loss: 0.4131, Train Acc: 81.50% | Val Loss: 0.4534, Val Acc: 78.75%
Epoch [8/25] Train Loss: 0.4094, Train Acc: 80.84% | Val Loss: 0.4485, Val Acc: 78.36%
Epoch [9/25] Train Loss: 0.4098, Train Acc: 80.48% | Val Loss: 0.4561, Val Acc: 76.61%
Epoch [10/25] Train Loss: 0.4132, Train Acc: 81.32% | Val Loss: 0.4454, Val Acc: 77.39%
Epoch [11/25] Train Loss: 0.3988, Train Acc: 81.59% | Val Loss: 0.4509, Val Acc: 78

In [ ]:
test_img = "/content/image4.jpg"
if os.path.exists(test_img):
    result = predict_image(test_img)
    print(f"Prediction for {test_img}: {result}")
else:
    print("No test image found. Place a test.jpg in working directory.")

Prediction for /content/image4.jpg: out


In [ ]:
!pip install --upgrade sympy
!pip install --upgrade torch torchvision